# Annotations

This notebook demonstrates the annotation API for the Gen graph widget.
Annotations can be shown in two ways:

- **Inline annotations** added with `add_annotation()` are rendered directly on the graph — each
  annotation is tinted on the nodes it covers and labelled below its bounding box.
- **Annotation tracks** added with `add_annotation_track()` are rendered as aligned rows in a
  separate panel below the graph.

Track panels accept annotations from:
- A list of **`Annotation`** objects (each wraps a `GraphLocus` with a name)
- A database **group** (added via `repo.add_annotation()`) — `group=` keyword
- A **translated** GFF3 or BED file — `file=` keyword

**API summary:**
```python
# Build named annotations from search hits
a = Annotation(locus, name="label")           # single locus
a = Annotation([locus1, locus2], name="span") # multi-locus (merged segments)

# Annotation tracks (separate aligned panel below the graph)
widget.add_annotation_track([a1, a2], name="track label")
widget.add_annotation_track(group="my_group")
widget.add_annotation_track(file="path.gff", name="label")
widget.annotation_tracks()
widget.remove_annotation_track("track label")

# Inline annotations (drawn directly on the graph canvas)
widget.add_annotation(a1)
widget.add_annotation(a2)
widget.inline_annotations()
widget.remove_annotation("label")
widget.clear_all_inline_annotations()

# Clear everything
widget.clear_all_annotations()   # removes tracks AND inline annotations
```

## Setup

Import the Anderson promoter GFA.  Each promoter shares a set of conserved
sequence blocks (nodes), making this a convenient graph for demonstrating
span-level annotations.

In [1]:
import pathlib
import tempfile

import gen

REPO_ROOT = pathlib.Path(gen.__file__).parents[3]
GFA = REPO_ROOT / "fixtures" / "anderson_promoters.gfa"
assert GFA.exists(), f"Fixture not found: {GFA}"

repo = gen.Repository(str(pathlib.Path(tempfile.mkdtemp(prefix="gen-annotations-"))))
repo.import_gfa(str(GFA), 'anderson', 'pooled')

bgs = repo.get_block_groups()
bg = bgs[0]
print(f"{len(bgs)} block group(s) imported")

1 block group(s) imported


In [2]:
parts = REPO_ROOT / "fixtures" / "protein_segments.fa"
design = REPO_ROOT / "fixtures" / "protein_layout.csv"

repo.import_library_files('shuffling',str(parts),str(design) )

"Library 'shuffling' imported."

## Search for conserved motifs

`bg.search()` returns a list of `GraphLocus` objects — one per match.
Each locus knows exactly which blocks it spans and the byte offsets within
the boundary blocks.

We look for two well-known conserved elements in Anderson promoters:
- **−35 box** (`TTGAC`) — transcription factor binding site
- **−10 box** (`CTAGCTCAGT`) — core promoter element

In [3]:


fig = bg.plot()
for h in bg.search("TGCTAGCTA"):
    fig.show(h)


In [4]:
minus35_hits = bg.search("ttgac")
minus10_hits = bg.search("ctagctcagt")


print(f"−35 box hits : {len(minus35_hits)}")
print(f"−10 box hits  : {len(minus10_hits)}")

fig = bg.plot()
for h in minus35_hits:
    fig.show(h)
for h in minus10_hits:
    fig.show(h)

−35 box hits : 1
−10 box hits  : 1


## Track panel annotations

Build `Annotation` objects from search hits, then pass them to
`add_annotation_track`.  Each `Annotation` carries its own display name, so
multiple hits in the same track can have different labels.

In [5]:
fig2 = bg.plot(rows=20)

promoter_track = [gen.Annotation(h, name=f"-35 box #{i}") 
                  for i, h in enumerate(minus35_hits)]
promoter_track += [gen.Annotation(h, name=f"-10 box #{i}") 
                   for i, h in enumerate(minus10_hits)]

fig2.add_annotation_track(promoter_track, name="promoter_features")

print("Track panels:", fig2.annotation_tracks())

Track panels: ['promoter_features']


In [6]:
fig2.show_path()

## Multiple annotation layers

Call `add_annotation_track` multiple times to stack layers.  Each layer gets
its own colour row.  Individual `Annotation` objects in each layer can have
distinct names.

In [7]:
fig4 = bg.plot(rows=24)

# Each layer is a list of Annotation objects with per-hit names.
fig4.add_annotation_track(
    [gen.Annotation(h, name="-35 box") for h in minus35_hits],
    name="-35 layer",
)
fig4.add_annotation_track(
    [gen.Annotation(h, name="-10 box") for h in minus10_hits],
    name="-10 layer",
)

print("Tracks:  ", fig4.annotation_tracks())

Tracks:   ['-35 layer', '-10 layer']


## Removing individual layers

`remove_annotation_track(name)` removes a single layer by the name that was
passed to `add_annotation_track`.  The widget re-renders immediately.

In [8]:
fig6 = bg.plot(rows=24)

fig6.add_annotation_track(
    [gen.Annotation(h, name="-35 box") for h in minus35_hits],
    name="-35 box",
)
fig6.add_annotation_track(
    [gen.Annotation(h, name="-10 box") for h in minus10_hits],
    name="-10 box",
)

print("Before remove:", fig6.annotation_tracks())

fig6.remove_annotation_track("-35 box")

print("After  remove:", fig6.annotation_tracks())

Before remove: ['-35 box', '-10 box']
After  remove: ['-10 box']


## Clearing all annotations

`clear_all_annotations()` removes every track panel in one call and triggers
a re-render.

In [9]:
fig7 = bg.plot(rows=22)

fig7.add_annotation_track(
    [gen.Annotation(h, name="-35 box") for h in minus35_hits],
    name="-35 box",
)
fig7.add_annotation_track(
    [gen.Annotation(h, name="-10 box") for h in minus10_hits],
    name="-10 box",
)

print("Before clear:", fig7.annotation_tracks())

fig7.clear_all_annotations()

print("After  clear:", fig7.annotation_tracks())

Before clear: ['-35 box', '-10 box']
After  clear: []


## Freeze with annotations

Call `freeze()` after loading annotations to bake the annotated view into a
static PNG for distribution.  The PNG is embedded in the `.ipynb` file and
renders in GitHub, nbviewer, and other static viewers.

In [10]:
fig8 = bg.plot(rows=20)
fig8.add_annotation_track(
    [gen.Annotation(h, name="-35 box") for h in minus35_hits],
    name="-35 box",
)
fig8.add_annotation_track(
    [gen.Annotation(h, name="-10 box") for h in minus10_hits],
    name="-10 box",
)
for m in minus10_hits:
    fig8.show(m)  # auto-colour accent

#fig8.freeze()

## Inline annotations

`add_annotation()` renders an annotation **directly on the graph canvas**.
Each span is tinted with an accent colour and its name is placed just below
the span's bounding box.

Label placement rules:
- X: centred under the visible portion of the span (tracks the viewport as you pan)
- Y: one row below the lowest node of the bounding box; drops a row to avoid
  overlapping other labels, but gives up rather than overwrite graph content

In [11]:
fig9 = bg.plot(rows=28)

for i, h in enumerate(minus35_hits):
    fig9.add_annotation(gen.Annotation(h, name=f"−35:{i}"))
for i, h in enumerate(minus10_hits):
    fig9.add_annotation(gen.Annotation(h, name=f"−10:{i}"))

print("Inline annotations:", fig9.inline_annotations())

Inline annotations: ['−35:0', '−10:0']


### Removing inline annotations

`remove_annotation(name)` and `clear_all_inline_annotations()` mirror the
track panel API.  `clear_all_annotations()` removes both tracks and inline
annotations at once.

In [12]:
fig10 = bg.plot(rows=28)
for h in minus35_hits:
    fig10.add_annotation(gen.Annotation(h, name="-35 box"))
for h in minus10_hits:
    fig10.add_annotation(gen.Annotation(h, name="-10 box"))
print("Before:", fig10.inline_annotations())

fig10.remove_annotation("-35 box")
print("After:", fig10.inline_annotations())

Before: ['-35 box', '-10 box']
After: ['-10 box']


#### Removing annotations that share a name

`remove_annotation(name)` removes **all** inline annotations with that name.
If `add_annotation` was called multiple times with annotations that happen to
share the same name, one `remove_annotation` call clears all of them.

In [13]:
fig12 = bg.plot(rows=28)

# Two annotations share the name "-35 box" — one per hit.
for h in minus35_hits:
    fig12.add_annotation(gen.Annotation(h, name="-35 box"))
for h in minus10_hits:
    fig12.add_annotation(gen.Annotation(h, name="-10 box"))

print("Before:", fig12.inline_annotations())

# Removes every annotation named "-35 box", not just the first.
fig12.remove_annotation("-35 box")
print("After:", fig12.inline_annotations())

Before: ['-35 box', '-10 box']
After: ['-10 box']


### Overlapping inline annotations

When two inline annotations cover the same graph cell, the more recently added
annotation overwrites the earlier one.  The last annotation rendered takes precedence
for both the tint and label.  This is simpler and more predictable than trying
to show both colours.

Below we add a broad "-35 region" annotation (searching for the core `ttg` motif)
and a narrower "-35 box" annotation (the full `ttgac` consensus).  Because the
shorter motif is a subset of the longer one, the `-35 box` annotation
overwrites the `-35 region` tint on shared cells.

In [14]:
# Search for a broader motif that encompasses the -35 consensus.
minus35_broad = bg.search("ttg")  # superset of the "ttgac" sites

fig11 = bg.plot(rows=28)

# First: broad -35 region (tints the nodes first)
for i, h in enumerate(minus35_broad):
    fig11.add_annotation(gen.Annotation(h, name=f"-{i}"))

# Second: exact -35 box (overwrites the broad -35 region tint)
for i, h in enumerate(minus35_hits):
    fig11.add_annotation(gen.Annotation(h, name=f"-35 box:{i}"))

print("Inline annotations:", fig11.inline_annotations())
fig11

Inline annotations: ['-0', '-1', '-2', '-3', '-4', '-5', '-6', '-7', '-8', '-9', '-10', '-11', '-12', '-13', '-14', '-15', '-16', '-17', '-18', '-19', '-20', '-21', '-22', '-23', '-24', '-25', '-26', '-27', '-28', '-29', '-30', '-31', '-32', '-33', '-34', '-35', '-36', '-37', '-38', '-35 box:0']
